## Implémentation de strategies

### Helper

La stratégie ci-dessous vous donne quelques pistes pour résoudre quelques stratégies simples.

In [2]:
class IStrategy :

    def __init__(self, name="IStrategy", _isRandom=False):
        self.name = name
        self._isRandom = _isRandom

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        '''
        @return (action, amountOfSecurity)
        '''
        pass

In [3]:
import random


class StrategyHelper(IStrategy):

    def __init__(self):
        super().__init__("StrategyHelper", True) # code obligatoire

        # ajouter ici les données dont vous aurez besoin

        # self.prices est la liste des prix des unités depuis le début de la partie
        self.prices = []

        # self.ma20 est la moyenne mobile sur 20 jours
        self.ma20 = []
        # self.ma50 est la moyenne mobile sur 50 jours
        self.ma50 = []

        self.actionPool = [Game.STILL, Game.INVEST_ALL, Game.DESINVEST_ALL, Game.CASH_IN]
        

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        
        # exemple pour accéder au jour, mois, année
        print("date day/month/year", date.day, date.month, date.year)

        if(len(self.prices) > 20):
            # une série est créée à partir des prix, elle est découpée en paquet de 20 et on calcule la moyenne de chaque paquet
            self.ma20 = pd.Series(self.prices).rolling(window=20).mean().values

        if(len(self.prices) > 50):
            self.ma50 = pd.Series(self.prices).rolling(window=50).mean().values   

        self.prices.append(unityPrice)

        
        # sélection d'une action aléatoire
        action = random.choice(self.actionPool)

        return (action, 0)

## Invest

In [ ]:

class StrategyInvest(IStrategy):
    '''
    Stratégie d'investissement qui investit tout au début de chaque mois
    puis fait un cash in de 100 unités monétaires une fois dans le mois.

    Cette stratégie doit être utilisée sur une action que vous pensez être haussière sur le long terme.
    En pratique, pour limiter les risques, elle est utilisée sur un ETF répliquant un indice boursier majeur (ex: S&P500, MSCI World, etc).
    L'hypothèse sous jacente est que sur le long terme, les marchés boursiers sont haussiers.

    Votre stratégie spéculative doit surperformer cette stratégie.
    '''
    def __init__(self, amountInvest=100):
        super().__init__("StrategyInvest", False) # code obligatoire

        # ajouter ici les données dont vous aurez besoin

        # self.precMonth permet de sauvegarder le mois précédent
        self.precMonth = -1
        self.amountInvest = amountInvest

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        
        # exemple pour accéder au jour, mois, année
        # print("date day/month/year", date.day, date.month, date.year)

        actions = []
        currentMonth = date.month
        
        # chaque début de mois, on investit tout l'argent disponible
        # puis une fois dans le mois, on fait un cash in de 100 unités monétaires
        if(currentMonth != self.precMonth):
            self.precMonth = currentMonth
            actions.append((Game.CASH_IN, self.amountInvest))
            actions.append((Game.INVEST_ALL, 0))
        
        return actions


### LongTerm

In [ ]:
class StrategyLongTerm(IStrategy):
    """
    Docstring de con j'ai la flemme
    """

    def __init__(self, dateBuy):
        super().__init__("StrategyLongTerm", False) # code obligatoire
        self.dateBuy = dateBuy

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        actions = []    
        if date == self.dateBuy:
            actions.append((game.INVEST_ALL, 0))
        return actions

### Random

In [11]:
class StrategyRandom(IStrategy):
    """
    Docstring à faire lol
    """
    def __init__(self):
        super().__init__("StrategyRandom", False) # code obligatoire

    def getAction(self, game, unityPrice, account, date, unityData=None, debug=False):
        actions = []
        self.currentAction = None
        self.randomAction = random.randint(1,4)
        if self.randomAction == 1:
            self.currentAction = "INVEST"
            nbr = random.uniform(0, account.cash)
            actions.append((game.INVEST, nbr))
            
        elif self.randomAction == 2:
            self.currentAction = "DESINVEST"
            nbr = random.uniform(0, account.security)
            actions.append((game.DESINVEST, nbr))


        elif self.randomAction == 3:
            self.currentAction = "INVEST_ALL"
            actions.append((game.INVEST_ALL, 0))


        else:
            self.currentAction = "DESINVEST_ALL"
            actions.append((game.DESINVEST_ALL, 0))

        return actions[0]
        

        

### Oracle

### January